In [ ]:
# ---- SYMSOL-CHILE with the NATIVE STF flow (persistent per-token Cartesian STF_2..6 inside GATr) ----------------
# Model and recipe: exactly those of Kaggle kernel saudabubin/symsol-native-stf6-no-nc-t4x2 (2026-10-29; SYMSOL I 10 %:
# median 1.11 deg, mean of per-shape medians 1.67 deg), code recovered from that kernel's CODE_B64 archive and patched
# only to (a) read SYMSOL-CHILE (--symsol_set 5: pyramids with 7 / 10 / 13 sides, rendered in this kernel) and
# (b) write a resume checkpoint every epoch (engine/resume.py). Changes for one RTX Pro 6000: one process (--no-ddp) at
# the same global batch 64; 1,000 test views per shape (the symsol-chile protocol).
# Estimated wall time on one RTX Pro 6000: ~1.5-2 h (SYMSOL I took 9.5 h on 2xT4; CHILE has 60 % of the training
# images and 12 % of the test views). Over 30 minutes: launched with a human's approval.
# OFFLINE: attach the native-chile code snapshot (ga-research-native-chile.tar.gz) and the offline-deps datasets.
RUN_NAME = 'chile_10pct_native_stf6_spherecloud_no_nc_seed0'
WANDB_API_KEY = None
REPO = 'chagrygoris/GA-Research'
BRANCH = 'claude/symsol-symsol2-differences-2dukos'   # unused offline; the native code is not on any branch
ARGS = ['--model', 'clifford_flow', '--encoder', 'resnet50', '--dataset', 'symsol', '--vector_field', 'gatr',
        '--condition_head', 'none', '--cond_tokens', 'spherecloud', '--pose_tokens', 'frame_pin', '--gatr_native_stf',
        '--gatr_native_stf_lmax', '6', '--gatr_native_stf_channels', '8', '--no-gatr_zero_readout',
        '--gatr_native_stf_eval_every', '5', '--symsol_set', '5', '--symsol_train_views', '4500',
        '--symsol_test_views', '1000', '--no-use_warp', '--no-use_synth', '--num_workers', '4', '--no-ema',
        '--fixed_val_noise', '--batch_size', '64', '--optimizer', 'adamw', '--lr', '1e-4', '--lr_scaling', 'sqrt',
        '--lr_schedule', 'step', '--warmup_epochs', '0', '--n_epochs', '40', '--no-symsol_nearest_coupling',
        '--no-symsol_likelihood', '--no-ddp', '--seed', '0']
WATCHDOG_MINUTES = (240, 720)   # (RTX, other GPU): ~2x the estimate; a stop keeps the per-epoch resume checkpoint
MARKERS = {'pose3d/pose3d/models/native_stf.py': 'CartesianMaps', 'pose3d/pose3d/datasets/symsol.py': 'symsol_chile',
           'pose3d/pose3d/engine/resume.py': 'RESUMABLE_EXIT'}


In [ ]:
import glob, os, shutil, socket, subprocess, tarfile

def online(host='github.com', timeout=5):
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False

ONLINE = online()
# Fail fast without a GPU: an account whose competition grant is not active gets Kaggle's CPU image even though the
# push asked for an RTX Pro 6000 (seen on 2026-10-31), and the run would otherwise die minutes later in a CUDA call.
import torch
assert torch.cuda.is_available(), 'no CUDA GPU in this kernel (CPU image): the accelerator was not granted'
print('GPU:', torch.cuda.get_device_name(0))
print('internet:', ONLINE)
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
os.chdir('/kaggle/working')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
if markers:        # a repo snapshot dataset, auto-extracted by Kaggle
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    shutil.copytree(markers[0][:markers[0].index('/pose3d/pose3d/config.py')], '/kaggle/working/GA-Research')
elif tgzs:
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
else:
    assert ONLINE, 'no internet and no repo snapshot attached'
    r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, f'https://github.com/{REPO}.git'],
                       capture_output=True, text=True)
    print(r.stdout + r.stderr); r.check_returncode()
print(subprocess.run(['git', '-C', 'GA-Research', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)
for path, marker in MARKERS.items():
    assert marker in open(f'/kaggle/working/GA-Research/{path}').read(), f'stale code: {marker!r} is not in {path}'


In [ ]:
# Dependencies Kaggle lacks. Online: PyPI / GitHub, with the image's torch pinned by a constraints file.
# Offline: the wheels of the ga-research-offline-deps and ga-research-gatr-deps datasets (installed without
# dependencies: the image's torch must stay).
import importlib, importlib.metadata as md, sys
# ga-research-offline-deps holds the wheels and ResNet weights; ga-research-offline-deps-py313 adds the healpy
# wheel for the Python 3.13 image (October 2026). Every ga-research-offline-deps* folder attached is read.
deps = sorted(glob.glob('/kaggle/input/**/ga-research-offline-deps*', recursive=True))
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
if deps:
    wheels = sorted(w for d in deps for w in glob.glob(os.path.join(d, '*.whl')))
    if gatr_deps:
        wheels += sorted(glob.glob(os.path.join(gatr_deps[0], '*.whl')))
    # Keep only the wheels this image's Python can run: the datasets may hold one healpy build per Python version
    # (the Kaggle image moved from 3.12 to 3.13 in October 2026), and pip rejects the whole command on one bad wheel.
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags())
    wheels = [w for w in wheels if parse_wheel_filename(os.path.basename(w))[3] & supported]
    print('wheels:', [os.path.basename(w) for w in wheels])
    assert any('healpy' in w for w in wheels), 'no healpy wheel for this Python: attach a ga-research-offline-deps* dataset that has one'
    subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)
    ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(ckpt_dir, exist_ok=True)
    for w in glob.glob('/kaggle/input/**/resnet*-*.pth', recursive=True):   # ImageNet weights (resnet50, resnet101)
        shutil.copy(w, os.path.join(ckpt_dir, os.path.basename(w)))
else:
    assert ONLINE, 'no internet and no ga-research-offline-deps attached'
    cons = []
    for p in ['torch', 'torchvision', 'numpy', 'scipy', 'pillow', 'tqdm', 'wandb', 'scikit-image', 'matplotlib',
              'pandas', 'sympy', 'astropy', 'packaging']:
        try:
            cons.append(f'{p}=={md.version(p)}')
        except md.PackageNotFoundError:
            pass
    open('/kaggle/working/constraints.txt', 'w').write('\n'.join(cons) + '\n')
    subprocess.run(['pip', 'install', '-q', '-c', '/kaggle/working/constraints.txt', 'e3nn==0.5.9', 'healpy==1.19.0',
                    'git+https://github.com/chagrygoris/image2sphere.git',
                    'git+https://github.com/chagrygoris/clifford-group-equivariant-neural-networks.git'], check=True)
    # GATr, the vector field: --no-deps, its pins would replace torch.
    subprocess.run(['pip', 'install', '-q', '--no-deps', 'einops', 'opt_einsum',
                    'git+https://github.com/Qualcomm-AI-research/geometric-algebra-transformer.git'], check=True)
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford']:
    importlib.import_module(mod)
# GATr hard-imports xformers (only for attention masks the flow never passes); the repo installs a stub before
# importing it (models/gatr_denoiser._import_gatr), so check the import the way the training process does.
sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
from pose3d.models.gatr_denoiser import _import_gatr
_import_gatr()
print('python', sys.version.split()[0], '| deps import cleanly')


In [ ]:
# W&B: live when online (the wandb_api_key secret, or WANDB_API_KEY above); offline files + WANDB_SYNC lines otherwise.
key = WANDB_API_KEY
if key is None and ONLINE:
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('wandb_api_key')
    except Exception as e:   # no secret attached: fall back to offline logging
        print('wandb_api_key secret not available:', e)
if key and ONLINE:
    import wandb
    wandb.login(key=key)
    os.environ['WANDB_API_KEY'] = key
else:
    os.environ['WANDB_MODE'] = 'offline'
print('wandb mode:', os.environ.get('WANDB_MODE', 'online'))


In [ ]:
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
DATA = '/kaggle/working/data'
# Render SYMSOL-CHILE (seeded: the same images in every kernel). Skipped when an earlier cell run already did it.
if not os.path.exists(f'{DATA}/symsol_chile/train/rotations.npz'):
    subprocess.run([sys.executable, '-u', '-m', 'pose3d.datasets.symsol_chile', f'--root={DATA}', '--train', '4500',
                    '--test', '1000', '--device', 'cuda'], check=True)
print('SYMSOL-CHILE root:', DATA, flush=True)
# CPU preflight: the native model's own numerical tests (as the original kernel ran them), before any GPU time.
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests/test_native_stf.py',
                    'tests/test_symsol_chile.py', 'tests/test_resume.py'],
                   env={**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '4'}, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:], flush=True)
assert r.returncode == 0, 'CPU preflight failed'
name = RUN_NAME
# RESUME (engine/resume.py): the run writes /kaggle/working/<run_name>_resume.pth after every epoch, which Kaggle keeps
# as this kernel's output. To continue a stopped run, launch a new kernel with the stopped one attached as input
# (gpu_pool/launcher.py --kernel-source <owner>/<kernel>): its checkpoint is found here and passed as --resume.
resume = sorted(glob.glob(f'/kaggle/input/**/{name}_resume.pth', recursive=True))
resume_args = []
if resume:
    shutil.copy(resume[0], f'/kaggle/working/{name}_resume.pth')     # keep advancing the same file
    resume_args = [f'--resume=/kaggle/working/{name}_resume.pth']
    print('resuming from', resume[0], flush=True)
cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={DATA}', '--platform=kaggle', f'--run_name={name}',
       '--resume_dir=/kaggle/working', *resume_args, *ARGS]
print(' '.join(cmd), flush=True)
# The output is also written to /kaggle/working/<run_name>.log: Kaggle keeps /kaggle/working as the kernel's output
# even when its own log stream comes back empty (seen on long RTX sessions), so the per-epoch metrics survive.
# WATCHDOG: the training process is killed after MAX_MINUTES (WATCHDOG_MINUTES, about twice the estimate) so
# that a stuck session cannot burn the 12-hour GPU budget (two sessions did exactly that).
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
MAX_MINUTES = WATCHDOG_MINUTES[0 if 'RTX' in gpu_name else 1]
print(f'GPU: {gpu_name!r}, watchdog {MAX_MINUTES} min', flush=True)
import threading, time
with open(f'/kaggle/working/{name}.log', 'w') as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ})
    def _watchdog():
        time.sleep(MAX_MINUTES * 60)
        if proc.poll() is None:
            # SIGTERM first: the run exits with 75 and keeps its last epoch's resume checkpoint; SIGKILL only if stuck.
            print(f'WATCHDOG: stopping the run after {MAX_MINUTES} minutes (resumable)', flush=True)
            proc.terminate()
            try:
                proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                proc.kill()
    threading.Thread(target=_watchdog, daemon=True).start()
    for line in proc.stdout:
        print(line, end='', flush=True)
        fh.write(line); fh.flush()
    rc = proc.wait()
# 75: stopped (watchdog / SIGTERM) with a resume checkpoint in /kaggle/working; the kernel ends cleanly so its output
# is kept, and a new kernel with this one attached continues it. Anything else non-zero is a crash and fails the kernel.
if rc == 75:
    print(f'STOPPED, RESUMABLE: /kaggle/working/{name}_resume.pth; relaunch with this kernel as --kernel-source', flush=True)
else:
    assert rc == 0, f'pose3d exited with code {rc}'
